In [2]:
import os
import ast
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import numpy as np
from tqdm import tqdm

# ==========================================
# 1. Konfiguration & Pfade (aus deiner Analyse!)
# ==========================================
BASE_DIR = '/datasets/multi-view-pig-posture-recognition'

# Schalter für T1 oder T2 Training (Einfach auf 'T2' ändern für den zweiten Lauf)
MODE = 'T1' 

if MODE == 'T1':
    TRAIN_CSV = os.path.join(BASE_DIR, 'train1.csv')
    TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train1_images')
    SUBMISSION_PREFIX = 'T1_'
else:
    TRAIN_CSV = os.path.join(BASE_DIR, 'train2.csv')
    TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train2_images')
    SUBMISSION_PREFIX = 'T2_'

TEST_CSV = os.path.join(BASE_DIR, 'test.csv')
TEST_IMG_DIR = os.path.join(BASE_DIR, 'test_images')

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4

# GPU-Check (wie in deinem Notebook)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training läuft auf: {DEVICE} für {MODE}")

# ==========================================
# 2. Custom Dataset (mit deiner BBox-Logik)
# ==========================================
class PigPostureDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Bild laden (Nutzt jetzt sauber image_id wie in deiner Analyse)
        img_path = os.path.join(self.img_dir, row['image_id'])
        image = Image.open(img_path).convert("RGB")
        
        # --- Bounding Box Logik (aus deinem Notebook übernommen) ---
        bbox_data = row['bbox']
        if isinstance(bbox_data, str):
            bbox = ast.literal_eval(bbox_data)
        else:
            bbox = bbox_data
            
        xmin, ymin, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
        
        # Schwein ausschneiden
        crop_box = (xmin, ymin, xmin + w, ymin + h)
        image = image.crop(crop_box)
        
        # Transformieren (z.B. Resize auf 224x224)
        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, row['row_id']
        else:
            label = torch.tensor(row['class_id'], dtype=torch.long)
            return image, label

# Bild-Transformationen
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 3. Daten laden & Klassen-Gewichte berechnen
# ==========================================
train_dataset = PigPostureDataset(TRAIN_CSV, TRAIN_IMG_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

# Lösung für das Histogramm-Problem: Class Weights berechnen
labels = train_dataset.df['class_id'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

# ==========================================
# 4. Modell definieren (Transfer Learning)
# ==========================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5) # 5 Schweine-Klassen
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 5. Training Loop
# ==========================================
print("Starte Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    for images, labels in tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    # Macro F1 Score berechnen
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Loss: {running_loss/len(train_loader):.4f} | Train Macro-F1: {epoch_f1:.4f}")

# ==========================================
# 6. Inference (Vorhersage auf dem Test-Set)
# ==========================================
print("Starte Vorhersagen für die Submission...")
test_dataset = PigPostureDataset(TEST_CSV, TEST_IMG_DIR, transform=transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

model.eval()
submission_results = []

with torch.no_grad():
    for images, row_ids in tqdm(test_loader, desc="Testing"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        for row_id, pred in zip(row_ids, preds.cpu().numpy()):
            submission_results.append({'row_id': row_id, 'class_id': pred})

# ==========================================
# 7. Submission CSV generieren
# ==========================================
submission_df = pd.DataFrame(submission_results)
submission_filename = f"{SUBMISSION_PREFIX}baseline_model.csv"
submission_df.to_csv(submission_filename, index=False)
print(f"Fertig! Datei gespeichert als: {submission_filename}")

Training läuft auf: cuda für T1
Starte Training...


Epoche 1/10: 100%|██████████| 717/717 [04:13<00:00,  2.83it/s]


Loss: 0.5329 | Train Macro-F1: 0.7389


Epoche 2/10: 100%|██████████| 717/717 [04:01<00:00,  2.96it/s]


Loss: 0.2327 | Train Macro-F1: 0.8745


...

Loss: 0.0580 | Train Macro-F1: 0.9659


Epoche 10/10: 100%|██████████| 717/717 [02:11<00:00,  5.46it/s]


Loss: 0.0598 | Train Macro-F1: 0.9657
Starte Vorhersagen für die Submission...


Testing: 100%|██████████| 366/366 [00:42<00:00,  8.53it/s]

Fertig! Datei gespeichert als: T1_baseline_model.csv
